In [ ]:
import pandas as pd

df = pd.read_parquet("hf://datasets/ButterChicken98/plantvillage-image-text-pairs/data/train-00000-of-00001.parquet")

In [ ]:
df

In [ ]:
df = df.drop(columns= ["image"])

In [ ]:
df

In [ ]:
df = df.explode("captions", ignore_index=True)
df

In [ ]:
df["caption"].unique().tolist()

In [ ]:
len(df["caption"].unique().tolist())

In [ ]:
df.to_csv("plantvillage_TextDataset.csv", index=False)

In [ ]:
df = pd.read_csv("plantvillage_TextDataset.csv")
df

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [ ]:
encoder = LabelEncoder()
encoder.fit(df["caption"].tolist())
df["label"] = encoder.transform(df["caption"].tolist())
df

In [ ]:
df_train, df_test = train_test_split(df, test_size=0.8)

In [ ]:
from datasets import Dataset

In [ ]:
train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)

In [ ]:
!pip install transformers

In [ ]:
from transformers import AutoTokenizer

In [ ]:
model_name = "distilbert-base-uncased"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def mapper_function(data):
  return tokenizer(data['captions'], truncation=True)

In [ ]:
tokenized_train = train_dataset.map(mapper_function, batched=True)
tokenized_test = test_dataset.map(mapper_function, batched=True)

In [ ]:
from transformers import AutoModelForSequenceClassification

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=15)

In [ ]:
!pip install evaluate

In [ ]:
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
import evaluate
import numpy as np

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer)

In [ ]:
eval_metric = evaluate.load("accuracy")

In [ ]:
def compute_metrics(pred):
  logits, labels = pred
  prediction = np.argmax(logits, axis=1)
  return eval_metric.compute(predictions=prediction, references=labels)

In [ ]:
training_arguments = TrainingArguments(
    output_dir="checkpoints",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=1e-4,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_strategy="epoch",
    save_strategy="steps", # Change save_strategy to "steps"
    save_steps=500, # Save every 500 steps
    save_total_limit=2,
    report_to="none"
    )

In [ ]:
trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset = tokenized_train,
    eval_dataset = tokenized_test,
    tokenizer= tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
    )

In [ ]:
trainer.train()

In [ ]:
# Save the model and tokenizer to a folder called 'saved_model'
trainer.save_model("saved_model")
tokenizer.save_pretrained("saved_model")

In [ ]:
trainer.save_model("/content/drive/MyDrive/saved_model")
tokenizer.save_pretrained("/content/drive/MyDrive/saved_model")


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "saved_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)


In [ ]:
text = ["This is a sample caption."]
inputs = tokenizer(text, truncation=True, padding=True, return_tensors="pt")
outputs = model(**inputs)
predictions = outputs.logits.argmax(dim=1)
print("Predicted label:", predictions.item())


In [ ]:
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


df = pd.read_parquet("hf://datasets/ButterChicken98/plantvillage-image-text-pairs/data/train-00000-of-00001.parquet")

print(df.columns)
print(df.head())
class_column_name = 'caption'
class_names = df[class_column_name].unique().tolist()
print("Class names extracted:")
print(class_names)
model_path = "saved_model"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)
def predict(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        predicted_idx = torch.argmax(outputs.logits, dim=1).item()
    return class_names[predicted_idx]
sample_text = "image of a healthy plant leaf"
prediction = predict(sample_text)
print(f"Predicted class: {prediction}")


In [ ]:
!zip -r saved_model.zip saved_model